# Wave gen+cost — Pelamis & RM3, full East Coast, 2011–2020

Prefilters sites once (depth 10–2000 m, distance-to-shore < 200 km), then evaluates
each device across all years. Uses your `ReadTurbineData` + cost functions unchanged;
the only inlined part is a memory-safe nearest-bin match (searchsorted, verified
identical to the tool's argmin) so it runs at East-Coast scale instead of OOM-ing.

Output matches the tool's `GenPU_*.npz` key format, written to Tech Outputs/Wave.

In [1]:
# ================= CONFIG =================
import os, glob, zipfile, time
import numpy as np
from numpy.lib import format as npformat

os.chdir(r"C:\Users\rmiller9\Documents\East Coast Model")
from WaveDeviceTools_EastCoast import ReadTurbineData, ComputeAnnualCost_Pelamis, ComputeAnnualCost_RM3
from GeneralGeoTools_EastCoast import GetDistanceToShore

INPUT_DATA_PATH = r"C:\Users\rmiller9\Documents\East Coast Model"
WAVE_DIR   = INPUT_DATA_PATH + r"\Resource Data\Wave\EastCoast"
DESIGN_DIR = INPUT_DATA_PATH + r"\Tech Designs\Wave"
OUTPUT_DIR = INPUT_DATA_PATH + r"\Tech Outputs\Wave"
COASTLINE  = INPUT_DATA_PATH + r"\Geospatial Data\CoastLine\ne_10m_coastline.shp"
RES_TAG    = "_0.01degrees"           # which resource files to read

YEARS = list(range(2011, 2021))       # 2011..2020 inclusive
DEVICES = [                           # (csv name, cost function, TurbineName label)
    ("Pelamis.csv", ComputeAnnualCost_Pelamis, "Pelamis"),
    ("RM3.csv",     ComputeAnnualCost_RM3,     "RM3"),
]

MIN_DEPTH, MAX_DEPTH = 10.0, 2000.0   # m
MAX_DIST_KM          = 200.0          # km to shore
DISCOUNT             = 1
WAKE                 = 0.95           # matches GetEnergyPu
os.makedirs(OUTPUT_DIR, exist_ok=True)

def year_file(y): return os.path.join(WAVE_DIR, f"EastCoast_Wave_{y}{RES_TAG}.npz")
print("years:", YEARS[0], "-", YEARS[-1], "| devices:", [d[2] for d in DEVICES])
for y in YEARS:
    assert os.path.exists(year_file(y)), f"missing {year_file(y)}"
print("all resource files present")

years: 2011 - 2020 | devices: ['Pelamis', 'RM3']
all resource files present


## Helpers — header scan + memory-safe column streaming

In [2]:
def _hdr(fp):
    v = npformat.read_magic(fp)
    try: return npformat._read_array_header(fp, v)
    except AttributeError:
        return npformat.read_array_header_1_0(fp) if v==(1,0) else npformat.read_array_header_2_0(fp)

def npz_members(path):
    out={}
    with zipfile.ZipFile(path) as zf:
        for zi in zf.infolist():
            if zi.filename.endswith(".npy"):
                with zf.open(zi) as f: shape,fort,dt=_hdr(f)
                out[zi.filename[:-4]]=(shape,dt)
    return out

def stream_columns(path, key, keep_cols, row_block=256):
    keep_cols=np.asarray(keep_cols)
    with zipfile.ZipFile(path) as zf, zf.open(key+".npy") as f:
        shape,fort,dt=_hdr(f); T,S=shape; it=dt.itemsize
        out=np.empty((T,keep_cols.size),dtype=dt)
        for r0 in range(0,T,row_block):
            r1=min(r0+row_block,T)
            blk=np.frombuffer(f.read((r1-r0)*S*it),dtype=dt).reshape(r1-r0,S)
            out[r0:r1,:]=blk[:,keep_cols]
    return out

def nearest_bin(x, bins):
    """Nearest-bin index for sorted ascending bins. searchsorted-on-midpoints;
    verified identical to argmin(|x-bins|)."""
    edges = (bins[:-1] + bins[1:]) / 2.0
    return np.clip(np.searchsorted(edges, x), 0, len(bins)-1)

## Prefilter sites once (depth + distance)

Coordinates/depth are identical across years, so this runs once and is reused for
both devices. Distance-to-shore is computed on the depth-passing sites only.

In [3]:
D = np.load(year_file(YEARS[0]), allow_pickle=True)
coords_all = D["coordinates"].astype(np.float64)
depth_all  = np.abs(D["depth"].astype(np.float64))
D.close()
n_all = len(depth_all)

depth_mask = (depth_all >= MIN_DEPTH) & (depth_all <= MAX_DEPTH)
depth_idx  = np.flatnonzero(depth_mask)
print(f"depth [{MIN_DEPTH},{MAX_DEPTH}]m: {n_all:,} -> {depth_idx.size:,}")

print("computing distance-to-shore on depth-passing sites (once)...")
t0=time.time()
dist = np.asarray(GetDistanceToShore(INPUT_DATA_PATH, coords_all[depth_idx],
                                     CoastlineShpPath=COASTLINE), dtype=float)
keep_idx = depth_idx[dist < MAX_DIST_KM]
print(f"dist < {MAX_DIST_KM}km: {depth_idx.size:,} -> {keep_idx.size:,}  ({time.time()-t0:.0f}s)")

LatLong_keep = coords_all[keep_idx].astype(np.float32)
Depth_keep   = depth_all[keep_idx]
Dist_keep    = dist[dist < MAX_DIST_KM]
n_keep = keep_idx.size

# concatenated time index across all years (naive datetimes, passed through)
TimeList = np.concatenate([np.load(year_file(y), allow_pickle=True)["time_index"] for y in YEARS])
year_len = [len(np.load(year_file(y), allow_pickle=True)["time_index"]) for y in YEARS]
T_total  = int(sum(year_len))
print(f"KEEP {n_keep:,} sites | {T_total:,} timesteps across {len(YEARS)} years")

depth [10.0,2000.0]m: 425,775 -> 256,486
computing distance-to-shore on depth-passing sites (once)...

 Calculating distance to shore for viable site locations


100%|██████████| 12825/12825 [20:57<00:00, 10.20it/s]


dist < 200.0km: 256,486 -> 222,768  (1278s)
KEEP 222,768 sites | 29,224 timesteps across 10 years


## Evaluate each device

Streams each year's Hs/Tp for the kept sites, matches bins the memory-safe way,
writes `Energy_pu` and `RawResource` into disk-backed arrays year by year, then
computes cost + LCOE with your functions and saves in the tool's format.

In [4]:
def evaluate_device(csv_name, cost_func, label):
    save_path = os.path.join(OUTPUT_DIR, f"GenPU_{os.path.splitext(csv_name)[0]}_EastCoast_{YEARS[0]}_{YEARS[-1]}.npz")
    if os.path.exists(save_path):
        print(f"[skip] {os.path.basename(save_path)} exists"); return
    print("="*70); print(f"DEVICE: {label}   ({csv_name})"); print("="*70)

    T = ReadTurbineData(os.path.join(DESIGN_DIR, csv_name))
    RatedPower, E_Mec2El, E_Av, E_Tr = T["RatedPower"], T["E_Mec2El"], T["E_Av"], T["E_Tr"]
    Hs_Bins, Te_Bins, MP = T["Hs_Bins"][:,0], T["Te_Bins"][:,0], T["MP_Matrix"]

    # disk-backed outputs (time, sites), float16 like the tool
    epu_path = os.path.join(OUTPUT_DIR, f"_tmp_epu_{label}.npy")
    raw_path = os.path.join(OUTPUT_DIR, f"_tmp_raw_{label}.npy")
    Energy_pu   = npformat.open_memmap(epu_path, mode="w+", dtype=np.float16, shape=(T_total, n_keep))
    RawResource = npformat.open_memmap(raw_path, mode="w+", dtype=np.float16, shape=(T_total, n_keep))

    t0=time.time(); off=0
    for y, yl in zip(YEARS, year_len):
        Hs = stream_columns(year_file(y), "significant_wave_height", keep_idx).astype(np.float32)
        Tp = stream_columns(year_file(y), "peak_period",             keep_idx).astype(np.float32)
        IdxHs = nearest_bin(Hs, Hs_Bins)
        IdxTp = nearest_bin(Tp, Te_Bins)
        EnergyProduction = np.minimum(MP[IdxHs, IdxTp] * E_Mec2El, RatedPower)
        epu = (EnergyProduction / RatedPower) * WAKE * E_Av * E_Tr
        Energy_pu[off:off+yl, :]   = epu.astype(np.float16)
        RawResource[off:off+yl, :] = (0.5 * Hs**2 * Tp).astype(np.float16)
        off += yl
        print(f"  {y}: {yl} steps  ({time.time()-t0:.0f}s)")

    # capacity factor per site (mean over time), then cost + LCOE (your cost fn)
    CF = np.asarray(Energy_pu, dtype=np.float32).mean(axis=0)
    CAPEX=np.empty(n_keep); OPEX=np.empty(n_keep); Ann=np.empty(n_keep)
    for i in range(n_keep):
        a, cx, ox = cost_func(Dist_keep[i], Depth_keep[i]*100/1000)  # mooring len km
        Ann[i], CAPEX[i], OPEX[i] = a/100, cx/100, ox/100
    CAPEX/=DISCOUNT; OPEX/=DISCOUNT; Ann/=DISCOUNT
    RatedPower_MW = RatedPower/1e6
    with np.errstate(divide="ignore", invalid="ignore"):
        LCOE = Ann*1e6 / (RatedPower_MW*365*24*CF)

    np.savez(save_path,
        ReadMe=f"{label} | EastCoast {YEARS[0]}-{YEARS[-1]} | depth[{MIN_DEPTH},{MAX_DEPTH}] dist<{MAX_DIST_KM}km",
        Energy_pu=np.asarray(Energy_pu), RatedPower=RatedPower_MW,
        LatLong=LatLong_keep, Depth=Depth_keep,
        DistanceShore=Dist_keep.astype(np.float16),
        CAPEX_site=CAPEX.astype(np.float16), OPEX_site=OPEX.astype(np.float16),
        AnnualizedCost=Ann.astype(np.float16), RawResource=np.asarray(RawResource),
        TimeList=TimeList, NumberOfCellsPerSite=np.ones(n_keep),
        ResolutionKm=-1, ResolutionDegrees=0.01, LCOE=LCOE)

    del Energy_pu, RawResource
    os.remove(epu_path); os.remove(raw_path)
    sz=os.path.getsize(save_path)/1024**3
    print(f"saved {os.path.basename(save_path)}  {n_keep:,} sites, {sz:.2f} GB  "
          f"| mean CF {np.nanmean(CF):.3f} | median LCOE ${np.nanmedian(LCOE[np.isfinite(LCOE)]):.0f}/MWh\n")

for csv_name, cost_func, label in DEVICES:
    evaluate_device(csv_name, cost_func, label)
print("Done.")

DEVICE: Pelamis   (Pelamis.csv)
  2011: 2920 steps  (130s)
  2012: 2928 steps  (254s)
  2013: 2920 steps  (375s)
  2014: 2920 steps  (494s)
  2015: 2920 steps  (613s)
  2016: 2928 steps  (733s)
  2017: 2920 steps  (853s)
  2018: 2920 steps  (971s)
  2019: 2920 steps  (1093s)
  2020: 2928 steps  (1214s)
saved GenPU_Pelamis_EastCoast_2011_2020.npz  222,768 sites, 24.26 GB  | mean CF 0.300 | median LCOE $470/MWh

DEVICE: RM3   (RM3.csv)
  2011: 2920 steps  (116s)
  2012: 2928 steps  (232s)
  2013: 2920 steps  (346s)
  2014: 2920 steps  (461s)
  2015: 2920 steps  (575s)
  2016: 2928 steps  (690s)
  2017: 2920 steps  (806s)
  2018: 2920 steps  (917s)
  2019: 2920 steps  (1039s)
  2020: 2928 steps  (1163s)
saved GenPU_RM3_EastCoast_2011_2020.npz  222,768 sites, 24.26 GB  | mean CF 0.071 | median LCOE $3001/MWh

Done.


In [5]:
import zipfile, numpy as np, os
from numpy.lib import format as npformat

OUT = r"C:\Users\rmiller9\Documents\East Coast Model\Tech Outputs\Wave\GenPU_Pelamis_EastCoast_2011_2020.npz"

def _hdr(fp):
    v = npformat.read_magic(fp)
    try: return npformat._read_array_header(fp, v)
    except AttributeError:
        return npformat.read_array_header_1_0(fp) if v==(1,0) else npformat.read_array_header_2_0(fp)

print(os.path.basename(OUT), f"({os.path.getsize(OUT)/1024**3:.2f} GB)")
print("="*72)
with zipfile.ZipFile(OUT) as zf:
    for zi in sorted(zf.infolist(), key=lambda i: i.filename):
        if not zi.filename.endswith(".npy"): continue
        with zf.open(zi) as f: shape, fort, dt = _hdr(f)
        print(f"  {zi.filename[:-4]:22s} shape={str(shape):18s} {dt}")

# scalars + sanity stats (small arrays only)
D = np.load(OUT, allow_pickle=True)
print("\nscalars:")
for k in ["RatedPower","ResolutionDegrees","ResolutionKm"]:
    print(f"  {k:20s} = {D[k]}")
print(f"\n  sites            = {D['LatLong'].shape[0]:,}")
print(f"  timesteps        = {D['Energy_pu'].shape[0]:,}")
print(f"  lat range        = [{D['LatLong'][:,0].min():.2f}, {D['LatLong'][:,0].max():.2f}]")
print(f"  depth range      = [{D['Depth'].min():.1f}, {D['Depth'].max():.1f}] m")
print(f"  dist range       = [{D['DistanceShore'].min():.1f}, {D['DistanceShore'].max():.1f}] km")
lcoe = D['LCOE'][np.isfinite(D['LCOE'])]
print(f"  LCOE median/min  = ${np.median(lcoe):.0f} / ${lcoe.min():.0f} /MWh")
print(f"  time[0], time[-1]= {D['TimeList'][0]}, {D['TimeList'][-1]}")
D.close()

GenPU_Pelamis_EastCoast_2011_2020.npz (24.26 GB)
  AnnualizedCost         shape=(222768,)          float16
  CAPEX_site             shape=(222768,)          float16
  Depth                  shape=(222768,)          float64
  DistanceShore          shape=(222768,)          float16
  Energy_pu              shape=(29224, 222768)    float16
  LCOE                   shape=(222768,)          float64
  LatLong                shape=(222768, 2)        float32
  NumberOfCellsPerSite   shape=(222768,)          float64
  OPEX_site              shape=(222768,)          float16
  RatedPower             shape=()                 float64
  RawResource            shape=(29224, 222768)    float16
  ReadMe                 shape=()                 <U63
  ResolutionDegrees      shape=()                 float64
  ResolutionKm           shape=()                 int32
  TimeList               shape=(29224,)           object

scalars:
  RatedPower           = 1.5
  ResolutionDegrees    = 0.01
  ResolutionKm    